# TFG José Carlos Navarro — Detección de patrones dermoscópicos de BCC (últimas 5 capas de VGG16 descongeladas)

Se usa VGG16 preentrenada en ImageNet como extractor de características, descongelando sus últimas `NUM_CAPAS_DESCONGELADAS` capas para entrenarlas junto con la cabeza de clasificación (multi-etiqueta, 7 patrones dermoscópicos).

El notebook tiene tres partes independientes:

- **PARTE A — Entrenar el modelo**: carga las imágenes, construye el modelo y lo entrena de principio a fin. Solo hace falta ejecutarla una vez.
- **PARTE B — Cargar el modelo y hacer pruebas**: evalúa el modelo ya entrenado (umbral por patrón, matrices de confusión, criterio clínico BCC/No-BCC).
- **PARTE C — Grad-CAM**: mapas de activación para visualizar en qué se fija el modelo al predecir cada patrón.

Antes de cualquiera de las partes, ejecuta las celdas de configuración inicial (imports, detección de entorno, rutas).

Funciona en **Google Colab** y en **Kaggle Notebooks** (detecta automáticamente en cuál de los dos estás). Para Kaggle: sube tus imágenes como Dataset, añádelo con "+ Add Input", activa GPU en las opciones de la sesión, y ajusta `NOMBRE_DATASET_KAGGLE` en la celda de configuración.

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import PIL.Image as Image
import tensorflow as tf

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/input')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_DATOS = 'drive/MyDrive/TFG'    # imagenes de entrada
    RUTA_SALIDA = 'drive/MyDrive/TFG'   # modelo y datos ya calculados
elif IN_KAGGLE:
    # Kaggle monta los datasets en /kaggle/input/datasets/<usuario>/<nombre-dataset>/
    USUARIO_KAGGLE = 'jcnavarr0'
    NOMBRE_DATASET_KAGGLE = 'imagenestfg'
    RUTA_DATOS = os.path.join('/kaggle/input/datasets', USUARIO_KAGGLE, NOMBRE_DATASET_KAGGLE)  # solo lectura
    RUTA_SALIDA = '/kaggle/working'  # aqui si se puede escribir, y se guarda entre sesiones
else:
    # Ajusta esta ruta si ejecutas el notebook fuera de Colab/Kaggle (p.ej. en local)
    RUTA_DATOS = '../../IMAGENES_MEDICAS'
    RUTA_SALIDA = '../../IMAGENES_MEDICAS'

print("Colab:", IN_COLAB, " Kaggle:", IN_KAGGLE)
print("TensorFlow:", tf.__version__)

In [ ]:
# Rutas de las carpetas de datos e imagenes de prueba (solo lectura).
# Todas las carpetas cuelgan directamente de RUTA_DATOS (misma ruta).
RUTA_AUMENTADOS = os.path.join(RUTA_DATOS, 'datos_aumentados')
RUTA_RETICULO   = os.path.join(RUTA_DATOS, 'datos_test_reticulo')
RUTA_SIN_PATRON = os.path.join(RUTA_DATOS, 'datos_test_sinPatron')
RUTA_BCC   = os.path.join(RUTA_DATOS, 'BCC_ISIC')
RUTA_NOBCC = os.path.join(RUTA_DATOS, 'NoBCC_ISIC')

PATRONES = ['PigmentNetwork', 'Ulceration', 'Large_B_G_OvoidNests',
            'Multi_B_G_Globules', 'MapleLeaflike', 'SpokeWheel',
            'ArborizingTelangiectasia']

# Iniciales de cada patron, mismo orden que PATRONES, para listar de forma compacta
# los patrones disparados en una imagen (p.ej. en los fallos de la Seccion 6).
ABREVIATURAS_PATRONES = ['PN', 'ULC', 'OV', 'GL', 'ML', 'SW', 'AT']

IMG_SIZE = 256
UMBRAL_CLASIFICACION = 0.5
SEMILLA = 42
np.random.seed(SEMILLA)

# Cuantas capas finales de VGG16 se descongelan y se entrenan junto con la cabeza.
# El resto de VGG16 se queda congelada con los pesos de ImageNet.
NUM_CAPAS_DESCONGELADAS = 5

# datos_aumentados esta siempre incluida en el entrenamiento. Estas dos banderas
# controlan si se anaden tambien datos_test_reticulo y datos_test_sinPatron.
INCLUIR_RETICULO = True
INCLUIR_SINPATRON = True

# Si se aplica (o no) recorte centrado por porcentaje a cada grupo de imagenes de
# ENTRENAMIENTO. Si esta en False, esa carpeta se carga con la imagen completa.
# El recorte de BCC/NoBCC se define aparte en la Seccion 6, solo para evaluacion.
APLICAR_RECORTE_RETICULO = True
APLICAR_RECORTE_SINPATRON = True

PORCENTAJE_RECORTE = 0.6  # que porcentaje central se conserva al recortar datos_test_reticulo/datos_test_sinPatron

# Rutas de salida: modelo, historial y datos de test ya cargados. El nombre de fichero
# incluye la combinacion de datos de entrenamiento usada. No hay features en cache:
# los pesos de VGG16 cambian durante el entrenamiento, asi que su salida no se puede
# precalcular ni reutilizar entre epocas.
if INCLUIR_RETICULO:
    _sufijo_reticulo = 'conReticulo' + ('Recorte' if APLICAR_RECORTE_RETICULO else 'SinRecorte')
else:
    _sufijo_reticulo = 'sinReticulo'

if INCLUIR_SINPATRON:
    _sufijo_sinpatron = 'conSinPatron' + ('Recorte' if APLICAR_RECORTE_SINPATRON else 'SinRecorte')
else:
    _sufijo_sinpatron = 'sinSinPatron'

_SUFIJO_EXPERIMENTO = _sufijo_reticulo + '_' + _sufijo_sinpatron + '_ultimas5capas'

RUTA_MODELO     = os.path.join(RUTA_SALIDA, 'modelos_keras', 'TFG_JCNG_modelo_' + _SUFIJO_EXPERIMENTO + '.keras')
RUTA_HISTORIA   = os.path.join(RUTA_SALIDA, 'modelos_keras', 'TFG_JCNG_historia_' + _SUFIJO_EXPERIMENTO + '.json')
RUTA_TEST_DATOS = os.path.join(RUTA_SALIDA, 'modelos_keras', 'TFG_JCNG_test_' + _SUFIJO_EXPERIMENTO + '.npz')
os.makedirs(os.path.dirname(RUTA_MODELO), exist_ok=True)


def cargar_imagen(ruta):
    imagen = Image.open(ruta)
    imagen = imagen.convert('RGB')

    # Las imagenes de datos_aumentados ya vienen recortadas de cerca sobre el patron, y
    # nunca se les aplica ningun recorte adicional aqui.
    if RUTA_AUMENTADOS not in ruta:
        if (RUTA_BCC in ruta) or (RUTA_NOBCC in ruta):
            aplicar_recorte = APLICAR_RECORTE_BCCNOBCC
            porcentaje_recorte = PORCENTAJE_RECORTE_BCCNOBCC
        elif RUTA_RETICULO in ruta:
            aplicar_recorte = APLICAR_RECORTE_RETICULO
            porcentaje_recorte = PORCENTAJE_RECORTE
        else:
            aplicar_recorte = APLICAR_RECORTE_SINPATRON
            porcentaje_recorte = PORCENTAJE_RECORTE

        if aplicar_recorte:
            ancho, alto = imagen.size
            ancho_recorte = int(ancho * porcentaje_recorte)
            alto_recorte = int(alto * porcentaje_recorte)
            izquierda = (ancho - ancho_recorte) // 2
            arriba = (alto - alto_recorte) // 2
            imagen = imagen.crop((izquierda, arriba, izquierda + ancho_recorte, arriba + alto_recorte))

    if imagen.size != (IMG_SIZE, IMG_SIZE):
        imagen = imagen.resize((IMG_SIZE, IMG_SIZE))
    return np.asarray(imagen)

# PARTE A — Entrenar el modelo

Ejecuta esta parte sólo si no tienes ya un modelo entrenado guardado, o si quieres volver a entrenarlo desde cero (por ejemplo, si cambias las imágenes de entrada). Si ya entrenaste antes y sólo quieres hacer pruebas, puedes saltar directamente a la **PARTE B**, más abajo.

## 1. Juntar las imágenes y repartirlas en train/val/test

Leemos los nombres de fichero de `datos_aumentados` y decidimos la etiqueta (qué patrones tiene) de cada imagen a partir del propio nombre del fichero.

`INCLUIR_RETICULO` e `INCLUIR_SINPATRON` (celda de configuración) controlan, cada una por separado, si se añaden también `datos_test_reticulo` (etiqueta fija `PigmentNetwork = 1`) y `datos_test_sinPatron` (etiqueta todo ceros) al entrenamiento.

Después barajamos la lista y la partimos en 70% train / 20% val / 10% test.

In [ ]:
rutas_totales = []
etiquetas_totales = []

# 1) datos_aumentados: la etiqueta esta en el nombre del fichero
ficheros_aumentados = os.listdir(RUTA_AUMENTADOS)
for nombre in ficheros_aumentados:
    etiqueta = [0, 0, 0, 0, 0, 0, 0]
    for i in range(len(PATRONES)):
        if PATRONES[i] in nombre:
            etiqueta[i] = 1
    rutas_totales.append(os.path.join(RUTA_AUMENTADOS, nombre))
    etiquetas_totales.append(etiqueta)

print("Imagenes de datos_aumentados:", len(ficheros_aumentados))

if INCLUIR_RETICULO:
    # 2) datos_test_reticulo: todas tienen patron reticular (PigmentNetwork = 1)
    ficheros_reticulo = os.listdir(RUTA_RETICULO)
    for nombre in ficheros_reticulo:
        etiqueta = [1, 0, 0, 0, 0, 0, 0]
        rutas_totales.append(os.path.join(RUTA_RETICULO, nombre))
        etiquetas_totales.append(etiqueta)

    print("Imagenes de datos_test_reticulo:", len(ficheros_reticulo))
else:
    print("INCLUIR_RETICULO = False -> no se anade datos_test_reticulo al entrenamiento")

if INCLUIR_SINPATRON:
    # 3) datos_test_sinPatron: no tienen ningun patron BCC
    ficheros_sin_patron = os.listdir(RUTA_SIN_PATRON)
    for nombre in ficheros_sin_patron:
        etiqueta = [0, 0, 0, 0, 0, 0, 0]
        rutas_totales.append(os.path.join(RUTA_SIN_PATRON, nombre))
        etiquetas_totales.append(etiqueta)

    print("Imagenes de datos_test_sinPatron:", len(ficheros_sin_patron))
else:
    print("INCLUIR_SINPATRON = False -> no se anade datos_test_sinPatron al entrenamiento")

print("Total de imagenes:", len(rutas_totales))

# Juntamos ruta + etiqueta en pares, y barajamos el orden
datos_completos = list(zip(rutas_totales, etiquetas_totales))
np.random.shuffle(datos_completos)

total_imagenes = len(datos_completos)
num_train = round(0.7 * total_imagenes)
num_val = round(0.2 * total_imagenes)

lista_train = datos_completos[0:num_train]
lista_val = datos_completos[num_train:num_train + num_val]
lista_test = datos_completos[num_train + num_val:]

print("Train:", len(lista_train))
print("Val:", len(lista_val))
print("Test:", len(lista_test))

## 2. Cargar las imágenes en memoria

Esta celda puede tardar varios minutos con casi 6000 imágenes: el almacenamiento (Google Drive o el dataset de Kaggle) tarda un poco en abrir cada archivo. Las imágenes cargadas se quedan en memoria y se usan directamente en el entrenamiento (Sección 3).

In [ ]:
def cargar_lista_de_imagenes(lista):
    x = []
    y = []
    for ruta, etiqueta in lista:
        imagen = cargar_imagen(ruta)
        x.append(imagen)
        y.append(etiqueta)
    x = np.asarray(x)
    y = np.asarray(y)
    return x, y


print("Cargando imagenes de entrenamiento... (puede tardar varios minutos)")
train_set_x, train_set_y = cargar_lista_de_imagenes(lista_train)

print("Cargando imagenes de validacion...")
val_set_x, val_set_y = cargar_lista_de_imagenes(lista_val)

print("Cargando imagenes de test...")
test_set_x, test_set_y = cargar_lista_de_imagenes(lista_test)

print("Formas -> train_x:", train_set_x.shape, "val_x:", val_set_x.shape, "test_x:", test_set_x.shape)

## 3. Construir el modelo (VGG16 con las últimas 5 capas descongeladas) y entrenar

Se descongelan las últimas `NUM_CAPAS_DESCONGELADAS` capas de VGG16 (por defecto 5) y se entrenan junto con la cabeza (`GlobalAveragePooling2D` + `Dense(512)` + `Dropout` + `Dense` de salida), todo en un único modelo (`modelo_completo`) entrenado de principio a fin.

Se necesita GPU activa: al no poder cachear la salida de VGG16 (sus pesos cambian en cada época), cada época recorre la red entera. En Colab: `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`. En Kaggle: panel derecho > opciones de la sesión > Accelerator > GPU.

In [ ]:
input_shape = (IMG_SIZE, IMG_SIZE, 3)
vgg = tf.keras.applications.vgg16.VGG16(include_top=False, weights='imagenet', input_shape=input_shape)

# Congelamos toda VGG16 y despues descongelamos solo las ultimas NUM_CAPAS_DESCONGELADAS
# capas.
for capa in vgg.layers:
    capa.trainable = False
for capa in vgg.layers[-NUM_CAPAS_DESCONGELADAS:]:
    capa.trainable = True

x = tf.keras.layers.GlobalAveragePooling2D()(vgg.output)
x = tf.keras.layers.Dense(512, activation='relu')(x)
x = tf.keras.layers.Dropout(0.15)(x)
salida = tf.keras.layers.Dense(len(PATRONES), activation='sigmoid')(x)
modelo_completo = tf.keras.models.Model(inputs=vgg.input, outputs=salida)

optimizador = tf.keras.optimizers.Adam(learning_rate=0.00001, beta_1=0.9, beta_2=0.999, epsilon=1e-8, amsgrad=True)
modelo_completo.compile(loss='binary_crossentropy', optimizer=optimizador, metrics=['binary_accuracy'])
modelo_completo.summary()

In [ ]:
# VGG16 espera las imagenes preprocesadas igual que en ImageNet: conversion a BGR y
# resta de la media de cada canal.
train_set_x_prep = tf.keras.applications.vgg16.preprocess_input(train_set_x.astype('float32'))
val_set_x_prep = tf.keras.applications.vgg16.preprocess_input(val_set_x.astype('float32'))
test_set_x_prep = tf.keras.applications.vgg16.preprocess_input(test_set_x.astype('float32'))

EPOCHS = 20
BATCH_SIZE = 32

print("*Comienza el entrenamiento*")
historia = modelo_completo.fit(
    x=train_set_x_prep, y=train_set_y,
    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1,
    validation_data=(val_set_x_prep, val_set_y),
    shuffle=True,
)

modelo_completo.save(RUTA_MODELO)
with open(RUTA_HISTORIA, 'w') as f:
    json.dump(historia.history, f)

# Guardamos las imagenes de test (sin preprocesar, para ocupar menos) y sus etiquetas,
# para poder evaluar en la Parte B sin tener que repetir la carga de todas las imagenes.
np.savez(RUTA_TEST_DATOS, test_set_x=test_set_x, test_set_y=test_set_y)

print("Modelo guardado en:", RUTA_MODELO)

## 4. Curvas de entrenamiento

In [ ]:
if 'historia' in dir():
    # Acabamos de entrenar en esta misma sesion
    datos_historia = historia.history
else:
    # Cargamos el historial guardado en disco la ultima vez que se entreno
    with open(RUTA_HISTORIA, 'r') as f:
        datos_historia = json.load(f)

acc = datos_historia['binary_accuracy']
val_acc = datos_historia['val_binary_accuracy']
loss = datos_historia['loss']
val_loss = datos_historia['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Entrenamiento')
plt.plot(epochs_range, val_acc, label='Validación')
plt.legend(loc='lower right')
plt.title('Binary Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Entrenamiento')
plt.plot(epochs_range, val_loss, label='Validación')
plt.legend(loc='upper right')
plt.title('Loss')
plt.show()

# PARTE B — Cargar el modelo y hacer pruebas

Si ya entrenaste el modelo en otra sesión (Parte A) y sólo quieres ver resultados, no hace falta repetir nada de lo de arriba. Ejecuta únicamente las celdas de configuración inicial del principio del notebook (imports, detección de entorno, rutas) y luego la celda de abajo: recupera el modelo completo (VGG16 con las últimas capas descongeladas ya incluida) y las imágenes de test guardadas.

Si además quieres ver las curvas de entrenamiento sin haber entrenado en esta sesión, puedes volver a ejecutar la celda de la sección 4 ("Curvas de entrenamiento"): detecta que no acabas de entrenar y carga el historial guardado en disco automáticamente.

In [ ]:
if not os.path.exists(RUTA_MODELO) or not os.path.exists(RUTA_TEST_DATOS):
    print("No se encuentra el modelo o los datos de test guardados.")
    print("Tienes que ejecutar antes la PARTE A (entrenamiento) al menos una vez, en esta misma sesion o en otra.")
else:
    modelo_completo = tf.keras.models.load_model(RUTA_MODELO)

    datos_test_guardados = np.load(RUTA_TEST_DATOS)
    test_set_x = datos_test_guardados['test_set_x']
    test_set_y = datos_test_guardados['test_set_y']
    test_set_x_prep = tf.keras.applications.vgg16.preprocess_input(test_set_x.astype('float32'))

    print("Modelo y datos de test cargados correctamente desde", RUTA_SALIDA)

## 5. Clasificar el test aplicando un umbral (no `argmax`) y calcular sensibilidad/especificidad por patrón

Como los 7 patrones no son excluyentes (una lesión puede tener varios a la vez), se aplica un umbral (por defecto 0.5) a cada salida por separado, se cuentan verdaderos/falsos positivos y negativos, y se calcula la sensibilidad y especificidad de cada patrón.

In [ ]:
probs_test = modelo_completo.predict(test_set_x_prep)
pred_test = probs_test >= UMBRAL_CLASIFICACION

matrices_confusion_patrones = []  # lista de [tp, fp, tn, fn] por patron, mismo orden que PATRONES

for i in range(len(PATRONES)):
    patron = PATRONES[i]
    y_true = test_set_y[:, i]
    y_pred = pred_test[:, i]

    tp = 0
    fp = 0
    tn = 0
    fn = 0
    for j in range(len(y_true)):
        if y_pred[j] == 1 and y_true[j] == 1:
            tp = tp + 1
        elif y_pred[j] == 1 and y_true[j] == 0:
            fp = fp + 1
        elif y_pred[j] == 0 and y_true[j] == 0:
            tn = tn + 1
        elif y_pred[j] == 0 and y_true[j] == 1:
            fn = fn + 1

    matrices_confusion_patrones.append([tp, fp, tn, fn])

    if (tp + fn) > 0:
        sensibilidad = tp / (tp + fn)
    else:
        sensibilidad = None

    if (tn + fp) > 0:
        especificidad = tn / (tn + fp)
    else:
        especificidad = None

    print(patron)
    print("  TP =", tp, " FP =", fp, " TN =", tn, " FN =", fn)
    print("  Sensibilidad =", sensibilidad)
    print("  Especificidad =", especificidad)
    print("")

In [ ]:
# Matriz de confusion 2x2 de cada patron por separado (test interno), mismo orden que PATRONES
plt.figure(figsize=(16, 8))

for i in range(len(PATRONES)):
    tp, fp, tn, fn = matrices_confusion_patrones[i]
    matriz = [[tp, fn], [fp, tn]]

    plt.subplot(2, 4, i + 1)
    plt.imshow(matriz, cmap='Blues')
    plt.xticks([0, 1], ['Presente', 'Ausente'])
    plt.yticks([0, 1], ['Presente', 'Ausente'])
    plt.xlabel('Prediccion')
    plt.ylabel('Real')
    plt.title(PATRONES[i], fontsize=10)

    for fila in range(2):
        for columna in range(2):
            plt.text(columna, fila, str(matriz[fila][columna]), ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.show()

## 6. Comprobación específica: clasificación BCC vs No-BCC (carpetas `BCC_ISIC` / `NoBCC_ISIC`)

Se toman las 50 imágenes de `BCC_ISIC` y las 50 de `NoBCC_ISIC` y se pasan por el modelo. Una lesión se clasifica como BCC si **no** presenta patrón reticular (`PigmentNetwork = 0`) y presenta **al menos uno** de los otros 6 patrones. Se compara con la carpeta de origen (verdad de referencia) y se calculan accuracy, sensibilidad, especificidad y valor predictivo positivo.

`APLICAR_RECORTE_BCCNOBCC`/`PORCENTAJE_RECORTE_BCCNOBCC` se pueden cambiar y volver a ejecutar esta celda (y las siguientes: 6.1, 6.2 y la Parte C) sin reentrenar el modelo: solo afectan a la evaluación.

In [ ]:
# Cambia esto para elegir si se aplica recorte a las imagenes de BCC_ISIC/NoBCC_ISIC,
# sin reentrenar: solo afecta a las imagenes que se cargan a continuacion.
APLICAR_RECORTE_BCCNOBCC = True
PORCENTAJE_RECORTE_BCCNOBCC = 0.6  # que porcentaje central se conserva al recortar (solo si APLICAR_RECORTE_BCCNOBCC es True)

ficheros_bcc = os.listdir(RUTA_BCC)
ficheros_nobcc = os.listdir(RUTA_NOBCC)

imagenes_bccnobcc = []
nombres_bccnobcc = []
clases_bccnobcc = []       # clase clinica: BCC o No-BCC, segun la carpeta de origen
es_bcc_de_verdad = []       # 1 si la imagen es realmente BCC, 0 si no

for nombre in ficheros_bcc:
    ruta = os.path.join(RUTA_BCC, nombre)
    imagenes_bccnobcc.append(cargar_imagen(ruta))
    nombres_bccnobcc.append(nombre)
    clases_bccnobcc.append('BCC')
    es_bcc_de_verdad.append(1)

for nombre in ficheros_nobcc:
    ruta = os.path.join(RUTA_NOBCC, nombre)
    imagenes_bccnobcc.append(cargar_imagen(ruta))
    nombres_bccnobcc.append(nombre)
    clases_bccnobcc.append('No-BCC')
    es_bcc_de_verdad.append(0)

print("Cargadas", len(ficheros_bcc), "imagenes de BCC y", len(ficheros_nobcc), "de NoBCC")

x_bccnobcc = np.asarray(imagenes_bccnobcc)
x_bccnobcc_prep = tf.keras.applications.vgg16.preprocess_input(x_bccnobcc.astype('float32'))
probs_bccnobcc = modelo_completo.predict(x_bccnobcc_prep)
pred_patrones_bccnobcc = probs_bccnobcc >= UMBRAL_CLASIFICACION

indice_pigment_network = 0  # PigmentNetwork es el primer patron de la lista PATRONES

tp = 0
fp = 0
tn = 0
fn = 0

for i in range(len(nombres_bccnobcc)):
    tiene_reticular = pred_patrones_bccnobcc[i][indice_pigment_network]

    tiene_algun_otro_patron = False
    for j in range(1, len(PATRONES)):
        if pred_patrones_bccnobcc[i][j] == 1:
            tiene_algun_otro_patron = True

    # Criterio clinico: BCC = sin patron reticular + al menos uno de los otros 6 patrones presente.
    if (not tiene_reticular) and tiene_algun_otro_patron:
        prediccion_bcc = 1
    else:
        prediccion_bcc = 0

    verdad = es_bcc_de_verdad[i]

    if prediccion_bcc == 1 and verdad == 1:
        tp = tp + 1
    elif prediccion_bcc == 1 and verdad == 0:
        fp = fp + 1
    elif prediccion_bcc == 0 and verdad == 0:
        tn = tn + 1
    elif prediccion_bcc == 0 and verdad == 1:
        fn = fn + 1

    # Para cada patron disparado, mostramos tambien el porcentaje (probabilidad) que dio
    # el modelo, para ver de cerca lo seguro o inseguro que estaba al disparar cada uno.
    patrones_disparados = ''
    for j in range(len(PATRONES)):
        if pred_patrones_bccnobcc[i][j] == 1:
            porcentaje = round(probs_bccnobcc[i][j] * 100)
            if patrones_disparados != '':
                patrones_disparados = patrones_disparados + ','
            patrones_disparados = patrones_disparados + ABREVIATURAS_PATRONES[j] + '(' + str(porcentaje) + '%)'

    if prediccion_bcc != verdad:
        print("Fallo en la imagen:", nombres_bccnobcc[i], " (clase real:", clases_bccnobcc[i], ") patrones:", patrones_disparados)
    else:
        print("Acierto en la imagen:", nombres_bccnobcc[i], " (clase real:", clases_bccnobcc[i], ") patrones:", patrones_disparados)

print("")
print("TP =", tp, " FP =", fp, " TN =", tn, " FN =", fn)
print("Accuracy =", (tp + tn) / (tp + fp + tn + fn))
if (tp + fn) > 0:
    print("Sensibilidad =", tp / (tp + fn))
if (tn + fp) > 0:
    print("Especificidad =", tn / (tn + fp))
if (tp + fp) > 0:
    print("PPV =", tp / (tp + fp))

## 6.1 Diagnóstico: probar otros umbrales sin reentrenar

La regla de BCC exige detectar al menos uno de 6 patrones: si el umbral 0.5 es demasiado conservador, muchas imágenes BCC reales pueden quedar sin ningún patrón detectado. Se prueban varios umbrales sobre las probabilidades ya calculadas (`probs_bccnobcc`), sin volver a pasar las imágenes por la red.

## 6.2 Diagnóstico: ¿qué patrón conviene mirar primero con Grad-CAM?

Se cuenta, de las 100 imágenes de BCC/NoBCC, cuántas tienen cada patrón activado dentro de BCC y dentro de NoBCC, ordenados de más a menos disparado. El que más se dispara en NoBCC es candidato a falsos positivos; el que menos se dispara en BCC, a falsos negativos.

In [ ]:
for umbral_prueba in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    pred_patrones_prueba = probs_bccnobcc >= umbral_prueba

    tp_p = 0
    fp_p = 0
    tn_p = 0
    fn_p = 0

    for i in range(len(nombres_bccnobcc)):
        tiene_reticular = pred_patrones_prueba[i][indice_pigment_network]

        tiene_algun_otro_patron = False
        for j in range(1, len(PATRONES)):
            if pred_patrones_prueba[i][j] == 1:
                tiene_algun_otro_patron = True

        if (not tiene_reticular) and tiene_algun_otro_patron:
            prediccion_bcc = 1
        else:
            prediccion_bcc = 0

        verdad = es_bcc_de_verdad[i]

        if prediccion_bcc == 1 and verdad == 1:
            tp_p = tp_p + 1
        elif prediccion_bcc == 1 and verdad == 0:
            fp_p = fp_p + 1
        elif prediccion_bcc == 0 and verdad == 0:
            tn_p = tn_p + 1
        elif prediccion_bcc == 0 and verdad == 1:
            fn_p = fn_p + 1

    accuracy_p = (tp_p + tn_p) / (tp_p + fp_p + tn_p + fn_p)
    print("Umbral =", umbral_prueba, " Accuracy =", accuracy_p, " TP =", tp_p, " FP =", fp_p, " TN =", tn_p, " FN =", fn_p)

In [ ]:
# Cuantas imagenes de BCC y de NoBCC tienen activado cada patron
conteo_bcc = [0, 0, 0, 0, 0, 0, 0]
conteo_nobcc = [0, 0, 0, 0, 0, 0, 0]

for i in range(len(nombres_bccnobcc)):
    for j in range(len(PATRONES)):
        if pred_patrones_bccnobcc[i][j] == 1:
            if es_bcc_de_verdad[i] == 1:
                conteo_bcc[j] = conteo_bcc[j] + 1
            else:
                conteo_nobcc[j] = conteo_nobcc[j] + 1

# Ordenamos los patrones de mas a menos disparados (ordenacion simple, son solo 7)
orden_bcc = list(range(len(PATRONES)))
for a in range(len(orden_bcc)):
    for b in range(a + 1, len(orden_bcc)):
        if conteo_bcc[orden_bcc[b]] > conteo_bcc[orden_bcc[a]]:
            orden_bcc[a], orden_bcc[b] = orden_bcc[b], orden_bcc[a]

orden_nobcc = list(range(len(PATRONES)))
for a in range(len(orden_nobcc)):
    for b in range(a + 1, len(orden_nobcc)):
        if conteo_nobcc[orden_nobcc[b]] > conteo_nobcc[orden_nobcc[a]]:
            orden_nobcc[a], orden_nobcc[b] = orden_nobcc[b], orden_nobcc[a]

print("Patrones activados en imagenes BCC (de", len(ficheros_bcc), "), de mas a menos:")
for j in orden_bcc:
    print("  ", PATRONES[j], "->", conteo_bcc[j])

print("")
print("Patrones activados en imagenes NoBCC (de", len(ficheros_nobcc), "), de mas a menos:")
for j in orden_nobcc:
    print("  ", PATRONES[j], "->", conteo_nobcc[j])

## 6.3 Matriz de confusión

Visualizamos la matriz de confusión del criterio clínico BCC/No-BCC (los mismos `tp`/`fp`/`tn`/`fn` de la Sección 6, pero en formato de tabla 2x2).

In [ ]:
# Matriz de confusion 2x2 del criterio clinico BCC/No-BCC (umbral 0.5, Seccion 6)
matriz_confusion = [[tp, fn], [fp, tn]]

plt.figure(figsize=(5, 5))
plt.imshow(matriz_confusion, cmap='Blues')
plt.xticks([0, 1], ['BCC', 'No-BCC'])
plt.yticks([0, 1], ['BCC', 'No-BCC'])
plt.xlabel('Prediccion del modelo')
plt.ylabel('Clase real')
plt.title('Matriz de confusion BCC/No-BCC')

for fila in range(2):
    for columna in range(2):
        plt.text(columna, fila, str(matriz_confusion[fila][columna]), ha='center', va='center', fontsize=16)

plt.tight_layout()
plt.show()

# PARTE C — Mapas de activación (Grad-CAM)

Grad-CAM calcula el gradiente de la salida de un patrón respecto a los mapas de activación de una capa convolucional de VGG16: indica cuánto importa cada filtro para esa predicción. Se pondera cada filtro por su importancia, se suman todos y se aplica un ReLU. El resultado es un mapa de calor que se superpone sobre la imagen original.

VGG16 reduce la imagen en cada bloque: `block5_pool` da un mapa de 8x8, `block4_pool` de 16x16, `block3_pool` de 32x32. Cuanto más profunda la capa, más pixelado sale el mapa al agrandarlo, pero más relacionado está con la decisión final. Con `CAPA_GRAD_CAM`, abajo, se puede elegir la capa sin reentrenar nada.

Las imágenes de BCC/NoBCC usadas en esta parte son las mismas cargadas en la Sección 6, con el recorte según `APLICAR_RECORTE_BCCNOBCC`/`PORCENTAJE_RECORTE_BCCNOBCC`. El umbral de clasificación es siempre `UMBRAL_CLASIFICACION` (0.5).

In [ ]:
# Cambia esto para calcular el Grad-CAM con mas resolucion espacial, a costa de una capa
# menos ligada a la clasificacion final: 'block5_pool' (8x8, por defecto), 'block4_pool'
# (16x16) o 'block3_pool' (32x32).
CAPA_GRAD_CAM = 'block5_pool'

capa_conv_grad_cam = modelo_completo.get_layer(CAPA_GRAD_CAM)
modelo_grad_cam = tf.keras.models.Model(
    inputs=modelo_completo.input,
    outputs=[capa_conv_grad_cam.output, modelo_completo.output],
)


def hacer_grad_cam(imagen, indice_patron):
    imagen_prep = tf.keras.applications.vgg16.preprocess_input(imagen.astype('float32'))
    entrada = np.expand_dims(imagen_prep, axis=0)

    with tf.GradientTape() as cinta:
        mapas_conv, predicciones = modelo_grad_cam(entrada)
        salida_patron = predicciones[:, indice_patron]

    # Gradiente de la salida del patron respecto a los mapas de activacion de VGG16:
    # dice cuanto cambiaria la prediccion si cambiara un poco cada filtro.
    gradientes = cinta.gradient(salida_patron, mapas_conv)

    # Importancia de cada uno de los filtros: la media de su gradiente
    peso_por_filtro = tf.reduce_mean(gradientes[0], axis=(0, 1)).numpy()

    mapas_conv = mapas_conv[0].numpy()
    alto_mapa, ancho_mapa, num_filtros = mapas_conv.shape

    mapa_calor = np.zeros((alto_mapa, ancho_mapa))
    for filtro in range(num_filtros):
        mapa_calor = mapa_calor + peso_por_filtro[filtro] * mapas_conv[:, :, filtro]

    # Solo interesa lo que suma a favor del patron (ReLU), y normalizamos a [0, 1]
    mapa_calor = np.maximum(mapa_calor, 0)
    if mapa_calor.max() > 0:
        mapa_calor = mapa_calor / mapa_calor.max()

    return mapa_calor


def mostrar_grad_cam(imagen, mapa_calor, titulo):
    imagen_calor_grande = Image.fromarray(np.uint8(mapa_calor * 255))
    imagen_calor_grande = imagen_calor_grande.resize((IMG_SIZE, IMG_SIZE))
    mapa_calor_grande = np.asarray(imagen_calor_grande)

    plt.figure(figsize=(9, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(imagen)
    plt.title('Imagen original')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(imagen)
    plt.imshow(mapa_calor_grande, cmap='jet', alpha=0.5)
    plt.title(titulo, fontsize=9)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

## Probar Grad-CAM con cualquier otra imagen y patrón

La celda de abajo permite repetir el mismo Grad-CAM con cualquier imagen y cualquier patrón, sin tocar las funciones `hacer_grad_cam`/`mostrar_grad_cam`. Solo hay que cambiar `RUTA_IMAGEN_PRUEBA` y `PATRON_PRUEBA`:

- `RUTA_IMAGEN_PRUEBA` puede construirse con cualquiera de las carpetas ya cargadas (`RUTA_BCC`, `RUTA_NOBCC`, `RUTA_RETICULO`, `RUTA_SIN_PATRON`, `RUTA_AUMENTADOS`) más el nombre de un fichero de esa carpeta, o escribirse directamente como una ruta completa si quieres probar con una imagen que no esté en ninguna de esas carpetas.
- `PATRON_PRUEBA` tiene que ser exactamente uno de los siete nombres de `PATRONES` (por ejemplo `'PigmentNetwork'`, `'MapleLeaflike'`, etc.).

In [ ]:
# Cambia estas dos lineas para probar con otra imagen y otro patron
RUTA_IMAGEN_PRUEBA = os.path.join(RUTA_NOBCC, ficheros_nobcc[0])
# Patrones disponibles (copia y pega el nombre exacto tal cual):
# PigmentNetwork, Ulceration, Large_B_G_OvoidNests, Multi_B_G_Globules,
# MapleLeaflike, SpokeWheel, ArborizingTelangiectasia
PATRON_PRUEBA = 'ArborizingTelangiectasia'

imagen_prueba = cargar_imagen(RUTA_IMAGEN_PRUEBA)
indice_patron_prueba = PATRONES.index(PATRON_PRUEBA)

mapa_calor_prueba = hacer_grad_cam(imagen_prueba, indice_patron_prueba)
titulo_prueba = PATRON_PRUEBA + ' (' + os.path.basename(RUTA_IMAGEN_PRUEBA) + ')'
mostrar_grad_cam(imagen_prueba, mapa_calor_prueba, titulo_prueba)

## Ver los 7 patrones a la vez sobre la misma imagen

Calcula el Grad-CAM de los 7 patrones para la misma imagen (`RUTA_IMAGEN_PRUEBA`) y los muestra todos juntos. Si los 7 mapas salen casi idénticos, es señal de que el modelo no distingue espacialmente entre patrones.

In [ ]:
imagen_todos_patrones = cargar_imagen(RUTA_IMAGEN_PRUEBA)

plt.figure(figsize=(16, 8))

plt.subplot(2, 4, 1)
plt.imshow(imagen_todos_patrones)
plt.title('Imagen original\n' + os.path.basename(RUTA_IMAGEN_PRUEBA), fontsize=9)
plt.axis('off')

for i in range(len(PATRONES)):
    mapa_calor_i = hacer_grad_cam(imagen_todos_patrones, i)

    imagen_calor_grande = Image.fromarray(np.uint8(mapa_calor_i * 255))
    imagen_calor_grande = imagen_calor_grande.resize((IMG_SIZE, IMG_SIZE))
    mapa_calor_grande = np.asarray(imagen_calor_grande)

    plt.subplot(2, 4, i + 2)
    plt.imshow(imagen_todos_patrones)
    plt.imshow(mapa_calor_grande, cmap='jet', alpha=0.5)
    plt.title(PATRONES[i], fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

## Recorrer las imágenes de BCC/NoBCC con Grad-CAM por patrón

Recorre las 100 imágenes ya cargadas en la Sección 6 y muestra el Grad-CAM únicamente de las que tienen activado el patrón `PATRON_A_VISUALIZAR` (probabilidad >= `UMBRAL_CLASIFICACION`). Para cada una imprime el nombre de la imagen, si es BCC o No-BCC de verdad, la clase clínica real y la probabilidad que le dio el modelo. Sirve para comprobar, patrón a patrón, si cuando el modelo lo dispara está mirando realmente una estructura de ese tipo o no.

Cambia `PATRON_A_VISUALIZAR` para probar con cualquiera de los 7 patrones (mismos nombres que en `PATRONES`).

In [ ]:
# Cambia este valor para recorrer las imagenes de BCC/NoBCC con Grad-CAM
# Patrones disponibles (copia y pega el nombre exacto tal cual):
# PigmentNetwork, Ulceration, Large_B_G_OvoidNests, Multi_B_G_Globules,
# MapleLeaflike, SpokeWheel, ArborizingTelangiectasia
PATRON_A_VISUALIZAR = 'ArborizingTelangiectasia'

indice_patron_visualizar = PATRONES.index(PATRON_A_VISUALIZAR)

# Solo nos quedamos con las imagenes donde el patron esta activado
indices_activados = []
for i in range(len(nombres_bccnobcc)):
    if pred_patrones_bccnobcc[i][indice_patron_visualizar] == 1:
        indices_activados.append(i)

print("Patron:", PATRON_A_VISUALIZAR)
print("Activado en", len(indices_activados), "de", len(nombres_bccnobcc), "imagenes")
print("")

for i in indices_activados:
    probabilidad = round(probs_bccnobcc[i][indice_patron_visualizar], 2)

    if es_bcc_de_verdad[i] == 1:
        origen = 'BCC'
    else:
        origen = 'NoBCC'

    titulo = PATRON_A_VISUALIZAR + ' (' + nombres_bccnobcc[i] + ', ' + origen + ', clase: ' + clases_bccnobcc[i] + ', prob: ' + str(probabilidad) + ')'
    print(titulo)

    mapa_calor = hacer_grad_cam(imagenes_bccnobcc[i], indice_patron_visualizar)
    mostrar_grad_cam(imagenes_bccnobcc[i], mapa_calor, titulo)